### Sign Language Dictation (Landmark-First, Qwen3.5 + How2Sign Holistic)

This notebook implements a high-performance landmark pipeline with:
- Frontal-only MediaPipe Holistic landmarks from `PSewmuthu/How2Sign_Holistic`
- Shoulder-centered + scale normalization for signer invariance
- Uniform 32-frame sampling per clip
- Temporal-fusion Sign Adapter (Conv1D + 3-layer MLP with LayerNorm+GELU)
- Two-phase training: adapter warm-up, then QLoRA SFT on all layers
- BLEU-4 validation during training

In [1]:
import io
import math
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
import torch.nn as nn
import evaluate
from huggingface_hub import snapshot_download
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
    set_seed,
    logging as hf_logging,
 )
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

hf_logging.set_verbosity_error()
set_seed(3407)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

BASE_MODEL = "unsloth/Qwen3.5-9B"
DATASET_NAME = "PSewmuthu/How2Sign_Holistic"
FRAMES_PER_CLIP = 32
MISSING_FILL_VALUE = -10.0
MAX_TARGET_TOKENS = 64  # reduced from 96 for 3090 VRAM safety

# Phase 1: adapter-only alignment
WARMUP_STEPS = 750  # 500-1000 recommended
WARMUP_LR = 8e-4

# Phase 2: QLoRA SFT
SFT_EPOCHS = 1
SFT_LR = 1.5e-4
WEIGHT_DECAY = 0.01
BATCH_SIZE = 2         # micro-batch for 24GB cards
EVAL_BATCH_SIZE = 1    # generation is memory-heavy
GRAD_ACCUM = 16        # keep effective batch size
EVAL_EVERY_STEPS = 100
NUM_WORKERS = 0        # notebook-safe: avoids worker crashes with mmap/filesystem
PIN_MEMORY = (DEVICE == "cuda")

/home/tim/Documents/Master/Q3/MML/mml-sign-language/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


### 1) Data Loading + Landmark Preprocessing

We use **frontal-only** landmarks from `PSewmuthu/How2Sign_Holistic`, then apply:
1. Shoulder-centered normalization: subtract midpoint of left/right shoulders
2. Scale normalization: divide by shoulder distance
3. Missing-value fill: fixed sentinel (`-10.0`)
4. Uniform temporal sampling to exactly 32 frames

In [2]:
def uniform_sample_frames(arr: np.ndarray, n_frames: int = FRAMES_PER_CLIP) -> np.ndarray:
    t = arr.shape[0]
    if t == 0:
        raise ValueError("Empty landmark sequence")
    idx = np.linspace(0, t - 1, n_frames).round().astype(np.int64)
    return arr[idx]


def normalize_shoulders(
    arr: np.ndarray,
    left_shoulder_idx: int = 11,
    right_shoulder_idx: int = 12,
    fill_value: float = MISSING_FILL_VALUE,
    eps: float = 1e-6,
 ) -> np.ndarray:
    x = arr.copy()  # [T, K, C]

    # Mark invalid values
    invalid = ~np.isfinite(x)
    x[invalid] = np.nan

    k = x.shape[1]
    l_idx = left_shoulder_idx if left_shoulder_idx < k else 0
    r_idx = right_shoulder_idx if right_shoulder_idx < k else min(1, k - 1)

    left = x[:, l_idx, :]
    right = x[:, r_idx, :]

    shoulder_mid = (left + right) / 2.0  # [T, C]
    shoulder_dist = np.linalg.norm(left - right, axis=-1, keepdims=True)  # [T, 1]
    shoulder_dist = np.where(np.isfinite(shoulder_dist), shoulder_dist, np.nan)
    shoulder_dist = np.where(shoulder_dist < eps, np.nan, shoulder_dist)

    x = x - shoulder_mid[:, None, :]
    x = x / shoulder_dist[:, None, :]

    # Fill NaNs/Infs after normalization
    bad = ~np.isfinite(x)
    x[bad] = fill_value
    return x.astype(np.float32)


def flatten_landmarks(arr: np.ndarray) -> np.ndarray:
    # [T, K, C] -> [T, K*C]
    return arr.reshape(arr.shape[0], -1).astype(np.float32)

In [3]:
# local_dir = snapshot_download(
#     repo_id="PSewmuthu/How2Sign_Holistic",
# 	repo_type="dataset",
#     allow_patterns=[
#         "how2sign_holistic_features/train/frontal.rar",
#         "how2sign_holistic_features/val/frontal.rar",
#         "how2sign_holistic_features/test/frontal.rar",
# 		"*.csv"
#     ],
#     local_dir="./data"
# )

In [4]:
import os
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from typing import Any, Dict, List, Optional, Tuple
from tqdm import tqdm


class How2SignHolisticLandmarkDataset(Dataset):
    def __init__(
        self,
        root_dir: str,
        split: str,
        frames_per_clip: int = FRAMES_PER_CLIP,
        fill_value: float = MISSING_FILL_VALUE,
        cache_in_memory: bool = False,   # <- safer default
    ):
        self.root_dir = root_dir
        self.split = split
        self.frames_per_clip = frames_per_clip
        self.fill_value = fill_value
        self.cache_in_memory = cache_in_memory

        # ---- Load metadata ----
        meta_path = os.path.join(
            root_dir,
            "metadata",
            f"how2sign_realigned_{split}.csv"
        )
        self.meta = pd.read_csv(meta_path, sep="\t")

        # ---- Data directory ----
        view = "frontal"
        self.data_dir = os.path.join(root_dir, split, view)

        # ---- Build sample list ----
        files = set(os.listdir(self.data_dir))

        self.samples: List[Tuple[str, str]] = []

        for _, row in self.meta.iterrows():
            video_id = row["SENTENCE_NAME"]
            text = row["SENTENCE"]

            filename = f"{video_id}_holistic.npy"

            if filename in files:
                path = os.path.join(self.data_dir, filename)
                self.samples.append((path, text))

        print(f"[{split}] Found {len(self.samples)} samples")

        # ---- Optional caching ----
        self.cache: Optional[List[Tuple[np.ndarray, str]]] = None
        if cache_in_memory:
            self.cache = [
                self._process_file(path, text)
                for path, text in tqdm(self.samples, desc=f"Cache {split}")
            ]

    def _process_file(self, path: str, text: str) -> Tuple[np.ndarray, str]:
        # memory-mapped load (important for large dataset)
        arr = np.load(path, mmap_mode='r')

        arr = uniform_sample_frames(arr, self.frames_per_clip)
        arr = normalize_shoulders(arr, fill_value=self.fill_value)
        arr = flatten_landmarks(arr)

        return arr, text

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        if self.cache is not None:
            arr, text = self.cache[idx]
        else:
            path, text = self.samples[idx]
            arr, text = self._process_file(path, text)

        return {
            "landmarks": torch.tensor(arr, dtype=torch.float32),
            "text": text,
        }

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = How2SignHolisticLandmarkDataset(
	root_dir="./data/how2sign_holistic_features",
    split="test", # CHANGE TO TRAIN
    cache_in_memory=False,  # set True if RAM allows for maximum speed
 )
val_dataset = How2SignHolisticLandmarkDataset(
	root_dir="./data/how2sign_holistic_features",
    split="val",
    cache_in_memory=False,
 )

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

[test] Found 2343 samples
[val] Found 1739 samples
Train samples: 2343
Val samples: 1739


In [6]:
@dataclass
class Batch:
    landmarks: torch.Tensor
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor
    reference_texts: List[str]


class SignBatchCollator:
    def __init__(self, tokenizer, max_target_tokens: int = MAX_TARGET_TOKENS):
        self.tokenizer = tokenizer
        self.max_target_tokens = max_target_tokens

    def __call__(self, examples: List[Dict[str, Any]]) -> Batch:
        landmarks = torch.stack([e["landmarks"] for e in examples], dim=0)  # [B, T, D]
        texts = [e["text"] for e in examples]

        toks = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_target_tokens,
            return_tensors="pt",
            add_special_tokens=True,
        )

        labels = toks["input_ids"].clone()
        labels[toks["attention_mask"] == 0] = -100

        return Batch(
            landmarks=landmarks,
            input_ids=toks["input_ids"],
            attention_mask=toks["attention_mask"],
            labels=labels,
            reference_texts=texts,
        )


collator = SignBatchCollator(tokenizer)

train_loader_kwargs = dict(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collator,
 )
val_loader_kwargs = dict(
    dataset=val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collator,
 )

if NUM_WORKERS > 0:
    train_loader_kwargs["persistent_workers"] = True
    val_loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(**train_loader_kwargs)
val_loader = DataLoader(**val_loader_kwargs)

first_batch = next(iter(train_loader))
print(first_batch.landmarks.shape, first_batch.input_ids.shape)

torch.Size([2, 32, 1629]) torch.Size([2, 27])


In [7]:
class SignAdapter(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, llm_hidden_dim: int):
        super().__init__()
        self.temporal_conv = nn.Conv1d(
            in_channels=input_dim,
            out_channels=input_dim,
            kernel_size=3,
            padding=1,
            groups=1,
            bias=True,
        )
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, llm_hidden_dim),
        )

    def forward(self, landmarks: torch.Tensor) -> torch.Tensor:
        # landmarks: [B, T, D]
        x = landmarks.transpose(1, 2)    # [B, D, T]
        x = self.temporal_conv(x)
        x = x.transpose(1, 2)            # [B, T, D]
        x = self.mlp(x)                  # [B, T, H]
        return x


class LandmarkQwen(nn.Module):
    def __init__(self, base_model_name: str):
        super().__init__()
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=DTYPE,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        self.llm = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            trust_remote_code=True,
            quantization_config=bnb_config,
            device_map="auto",
        )
        self.llm.config.use_cache = False

        hidden_size = self.llm.config.hidden_size
        self.hidden_size = hidden_size

        # Infer landmark feature dimension from one sample
        sample_dim = train_dataset[0]["landmarks"].shape[-1]
        self.adapter = SignAdapter(
            input_dim=sample_dim,
            hidden_dim=min(2048, hidden_size),
            llm_hidden_dim=hidden_size,
        )

        self.prompt = "Transcribe the sign language sequence into fluent English:"
        prompt_ids = tokenizer(self.prompt, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
        self.register_buffer("prompt_ids", prompt_ids, persistent=False)

        # Keep adapter on same device as embedding layer
        emb_device = self.llm.get_input_embeddings().weight.device
        self.adapter.to(emb_device)

    def freeze_llm(self):
        for p in self.llm.parameters():
            p.requires_grad = False

    def unfreeze_lora_only(self):
        for p in self.llm.parameters():
            p.requires_grad = False
        for n, p in self.llm.named_parameters():
            if "lora_" in n:
                p.requires_grad = True

    def enable_qlora_all_linear(self, r: int = 16, alpha: int = 32, dropout: float = 0.05):
        self.llm = prepare_model_for_kbit_training(self.llm)
        lora_cfg = LoraConfig(
            r=r,
            lora_alpha=alpha,
            lora_dropout=dropout,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules="all-linear",
        )
        self.llm = get_peft_model(self.llm, lora_cfg)

        emb_device = self.llm.get_input_embeddings().weight.device
        self.adapter.to(emb_device)

    def _build_inputs(
        self,
        landmarks: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        bsz = landmarks.size(0)
        dev = self.llm.get_input_embeddings().weight.device

        landmarks = landmarks.to(dev)
        input_ids = input_ids.to(dev)
        attention_mask = attention_mask.to(dev)
        if labels is not None:
            labels = labels.to(dev)

        adapter_tokens = self.adapter(landmarks).to(dtype=self.llm.dtype)  # [B, T, H]
        prompt_ids = self.prompt_ids.unsqueeze(0).expand(bsz, -1).to(dev)
        prompt_embeds = self.llm.get_input_embeddings()(prompt_ids)
        text_embeds = self.llm.get_input_embeddings()(input_ids)

        inputs_embeds = torch.cat([adapter_tokens, prompt_embeds, text_embeds], dim=1)

        prefix_len = adapter_tokens.size(1) + prompt_embeds.size(1)
        prefix_mask = torch.ones((bsz, prefix_len), dtype=attention_mask.dtype, device=dev)
        full_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        out = {
            "inputs_embeds": inputs_embeds,
            "attention_mask": full_attention_mask,
        }
        if labels is not None:
            prefix_labels = torch.full((bsz, prefix_len), -100, dtype=labels.dtype, device=dev)
            out["labels"] = torch.cat([prefix_labels, labels], dim=1)
        return out

    def forward(self, batch: Batch):
        model_inputs = self._build_inputs(
            landmarks=batch.landmarks,
            input_ids=batch.input_ids,
            attention_mask=batch.attention_mask,
            labels=batch.labels,
        )
        return self.llm(**model_inputs)

    @torch.no_grad()
    def generate_text(self, landmarks: torch.Tensor, max_new_tokens: int = 96) -> List[str]:
        self.eval()
        bsz = landmarks.size(0)
        dev = self.llm.get_input_embeddings().weight.device

        landmarks = landmarks.to(dev)
        adapter_tokens = self.adapter(landmarks).to(dtype=self.llm.dtype)
        prompt_ids = self.prompt_ids.unsqueeze(0).expand(bsz, -1).to(dev)
        prompt_embeds = self.llm.get_input_embeddings()(prompt_ids)

        inputs_embeds = torch.cat([adapter_tokens, prompt_embeds], dim=1)
        attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=dev)

        out_ids = self.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        return tokenizer.batch_decode(out_ids, skip_special_tokens=True)


model = LandmarkQwen(BASE_MODEL)
print(f"LLM hidden size: {model.hidden_size}")

Loading weights: 100%|██████████| 427/427 [00:06<00:00, 64.17it/s, Materializing param=model.norm.weight]                               


LLM hidden size: 4096


In [8]:
bleu_metric = evaluate.load("bleu")

def evaluate_bleu4(model: LandmarkQwen, dataloader: DataLoader, max_batches: Optional[int] = 25) -> float:
    model.eval()
    preds, refs = [], []

    for step, batch in enumerate(tqdm(dataloader, desc="Validation BLEU-4", leave=False)):
        if max_batches is not None and step >= max_batches:
            break
        generated = model.generate_text(batch.landmarks, max_new_tokens=MAX_TARGET_TOKENS)
        generated = [g.strip() for g in generated]
        references = [r.strip() for r in batch.reference_texts]

        preds.extend(generated)
        refs.extend([[r] for r in references])

    score = bleu_metric.compute(predictions=preds, references=refs, max_order=4)
    return float(score["bleu"])


def cycle_loader(dataloader: DataLoader):
    while True:
        for batch in dataloader:
            yield batch


def move_batch_to_device(batch: Batch, device: str) -> Batch:
    return Batch(
        landmarks=batch.landmarks.to(device, non_blocking=True),
        input_ids=batch.input_ids.to(device, non_blocking=True),
        attention_mask=batch.attention_mask.to(device, non_blocking=True),
        labels=batch.labels.to(device, non_blocking=True),
        reference_texts=batch.reference_texts,
    )


# ---- Phase 1: VT-Align warm-up (adapter only) ----
model.freeze_llm()
for p in model.adapter.parameters():
    p.requires_grad = True

warmup_optim = torch.optim.AdamW(
    [p for p in model.adapter.parameters() if p.requires_grad],
    lr=WARMUP_LR,
    weight_decay=1e-4,
 )

model.train()
train_iter = cycle_loader(train_loader)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and DTYPE == torch.float16))
oom_skips = 0

progress = tqdm(range(1, WARMUP_STEPS + 1), desc="Phase 1/2: Adapter Warm-up")
for step in progress:
    batch = move_batch_to_device(next(train_iter), DEVICE)
    warmup_optim.zero_grad(set_to_none=True)

    try:
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda"), dtype=DTYPE):
            out = model(batch)
            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(warmup_optim)
        scaler.update()

        if step % 20 == 0 or step == 1:
            progress.set_postfix(loss=float(loss.item()), oom_skips=oom_skips)

    except torch.cuda.OutOfMemoryError:
        oom_skips += 1
        warmup_optim.zero_grad(set_to_none=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        progress.set_postfix(oom_skips=oom_skips, status="OOM skip")
        continue

warmup_bleu4 = evaluate_bleu4(model, val_loader, max_batches=25)
print(f"Phase 1 done | BLEU-4: {warmup_bleu4:.4f} | OOM skips: {oom_skips}")

Phase 1/2: Adapter Warm-up: 100%|██████████| 750/750 [03:56<00:00,  3.17it/s, loss=3.26, oom_skips=0]


KeyboardInterrupt: 

### 2) QLoRA SFT (All Layers: Attention + MLP)

Phase 2 unfreezes adapter + QLoRA modules and trains end-to-end using 4-bit base weights.
LoRA config: `r=16`, `alpha=32`, targeting **all linear layers**.

In [ ]:
model.enable_qlora_all_linear(r=16, alpha=32, dropout=0.05)
model.unfreeze_lora_only()
for p in model.adapter.parameters():
    p.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable params: {sum(p.numel() for p in trainable_params):,}")

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = SFT_EPOCHS * steps_per_epoch
warmup_steps = max(20, int(0.03 * total_steps))

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=SFT_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
 )
scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
 )

scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and DTYPE == torch.float16))
global_step = 0

In [ ]:
best_bleu4 = -1.0
history = []
oom_skips = 0

model.train()
for epoch in range(1, SFT_EPOCHS + 1):
	progress = tqdm(train_loader, desc=f"Phase 2/2: SFT Epoch {epoch}")
	optimizer.zero_grad(set_to_none=True)

	for step, batch in enumerate(progress, start=1):
		batch = move_batch_to_device(batch, DEVICE)

		try:
			with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda"), dtype=DTYPE):
				out = model(batch)
				loss = out.loss / GRAD_ACCUM

			scaler.scale(loss).backward()

			if step % GRAD_ACCUM == 0:
				scaler.step(optimizer)
				scaler.update()
				optimizer.zero_grad(set_to_none=True)
				scheduler.step()
				global_step += 1

				progress.set_postfix(
					loss=float(loss.item() * GRAD_ACCUM),
					lr=float(scheduler.get_last_lr()[0]),
					step=global_step,
					oom_skips=oom_skips,
				)

				if global_step % EVAL_EVERY_STEPS == 0:
					if torch.cuda.is_available():
						torch.cuda.empty_cache()
					# bleu4 = evaluate_bleu4(model, val_loader, max_batches=25)
					# history.append({"step": global_step, "bleu4": bleu4})
					history.append({"step": global_step, "bleu4": None})  # placeholder for BLEU-4 to save time
					# print(f"[Eval] step={global_step} | BLEU-4={bleu4:.4f}")
					print(f"[Eval] step={global_step} | BLEU-4=??? (skipped for speed)")
					model.train()

					# if bleu4 > best_bleu4:
					# 	best_bleu4 = bleu4
					# 	model.llm.save_pretrained("outputs/best_lora")
					# 	tokenizer.save_pretrained("outputs/best_lora")
					# 	torch.save(model.adapter.state_dict(), "outputs/best_sign_adapter.pt")

		except torch.cuda.OutOfMemoryError:
			oom_skips += 1
			optimizer.zero_grad(set_to_none=True)
			if torch.cuda.is_available():
				torch.cuda.empty_cache()
			progress.set_postfix(step=global_step, oom_skips=oom_skips, status="OOM skip")
			continue

print(f"Best BLEU-4: {best_bleu4:.4f} | OOM skips: {oom_skips}")

In [ ]:
# final_bleu4 = evaluate_bleu4(model, val_loader, max_batches=None)
# print(f"Final BLEU-4 (full val): {final_bleu4:.4f}")

# Show a few predictions
preview_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, collate_fn=collator)
preview_batch = next(iter(preview_loader))
pred_texts = model.generate_text(preview_batch.landmarks, max_new_tokens=MAX_TARGET_TOKENS)

for i, (pred, ref) in enumerate(zip(pred_texts, preview_batch.reference_texts), start=1):
    print(f"\nSample {i}")
    print("PRED:", pred)
    print("REF :", ref)

In [ ]:
# Save final artifacts
model.llm.save_pretrained("outputs/final_lora")
tokenizer.save_pretrained("outputs/final_lora")
torch.save(model.adapter.state_dict(), "outputs/final_sign_adapter.pt")

print("Saved:")
print("- outputs/final_lora")
print("- outputs/final_sign_adapter.pt")